# Regression: Shared Preprocessing Audit

**Owner: Samith**  
This notebook documents the cleaning, split, encoding, scaling, outlier treatment, and feature engineering used independently in every regression algorithm notebook.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
sns.set_theme(style="whitegrid", palette="colorblind")

data_path = os.path.join("..", "..", "data", "health_insurance.csv")
df = pd.read_csv(data_path).drop_duplicates().copy()
X = df.drop(columns="claim")
y = df["claim"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

# Cap plausible extremes using limits learned only from training predictors.
lower = X_train[numeric_features].quantile(0.01)
upper = X_train[numeric_features].quantile(0.99)
X_train = X_train.copy()
X_test = X_test.copy()
X_train[numeric_features] = X_train[numeric_features].clip(lower, upper, axis=1)
X_test[numeric_features] = X_test[numeric_features].clip(lower, upper, axis=1)

# Age and BMI can interact in their effect on insurance cost.
for frame in (X_train, X_test):
    frame["age_bmi_interaction"] = frame["age"] * frame["bmi"]
numeric_features.append("age_bmi_interaction")

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_features),
])

print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")

Training rows: 11,123; test rows: 2,781


In [2]:
missing_before = df[["age", "bmi"]].isna().sum()
prepared_train = preprocessor.fit_transform(X_train)
prepared_test = preprocessor.transform(X_test)

print("Missing values before train-only imputation:")
display(missing_before.to_frame("count"))
print("Prepared shapes:", prepared_train.shape, prepared_test.shape)
assert not np.isnan(prepared_train).any()
assert not np.isnan(prepared_test).any()

Missing values before train-only imputation:


,count
age,361
bmi,901


Prepared shapes: (11123, 147) (2781, 147)


Exact duplicates are removed before splitting. Missing numeric values use training medians. Numerical features are capped with training-set 1st/99th percentiles and standardised. Categoricals use most-frequent imputation and one-hot encoding. `age_bmi_interaction` is added because age can change how BMI relates to claim cost.